In [8]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

In [9]:
## DATA (CellOracle object before fitting GRN)
oracle = co.load_hdf5("../data/celloracle_data/celloracle_unfit.celloracle.oracle")

## FIT GRN — RIDGE REGRESSION PER CLUSTER

Use Links object to store raw GRNs per cluster (with weights and associated p-values):

In [10]:
# Step 1: infer GRN per cluster with Ridge Regression
# alpha: regularization strength. Higher = sparser network.
# verbose_level=10 prints progress per cluster
links = oracle.get_links(
    cluster_name_for_GRN_unit='leiden_annotated',
    alpha=10,
    verbose_level=10
)

# Save Links object
links.to_hdf5(file_path="../data/celloracle_data/celloracle_links_raw.celloracle.links")

  0%|          | 0/6 [00:00<?, ?it/s]

Inferring GRN for ASO_KO...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for Ectoderm_1...


  0%|          | 0/1941 [00:00<?, ?it/s]

...

  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for Mesoderm_2...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inferring GRN for mESCs...


  0%|          | 0/1941 [00:00<?, ?it/s]

Inspect the Links object:

In [13]:
links.links_dict.keys()

dict_keys(['ASO_KO', 'Ectoderm_1', 'Ectoderm_2', 'Mesoderm_1', 'Mesoderm_2', 'mESCs'])

In [15]:
links.links_dict["ASO_KO"]

,source,target,coef_mean,coef_abs,p,-logp
0,E2f2,1110051M20Rik,-0.004956,0.004956,1.581332e-01,0.800977
1,Klf16,1110051M20Rik,0.085048,0.085048,6.993005e-07,6.155336
2,Nfe2,1110051M20Rik,-0.010746,0.010746,3.024051e-03,2.519411
3,Klf1,1110051M20Rik,0.002166,0.002166,1.519522e-01,0.818293
4,Crebzf,1110051M20Rik,-0.014760,0.014760,1.886380e-03,2.724371
...,...,...,...,...,...,...
719588,Rarb,Zzef1,0.021106,0.021106,4.898473e-10,9.309939
719589,Tfap2a,Zzef1,0.011528,0.011528,1.918989e-07,6.716927
719590,Vdr,Zzef1,-0.005756,0.005756,9.833774e-04,3.007280
719591,Elf2,Zzef1,-0.003766,0.003766,3.479081e-02,1.458535


Filter the inferred GRNs by p-value (optional to also put a maximum of edges per cluster):

In [ ]:
# p: maximum adjusted p-value for the Ridge coefficient
# weight: rank edges by absolute coefficient value
# threshold_number: keep top N edges per cluster (not total) => by default 1e4
links.filter_links(
    p=0.000000000000000001,
    #, weight='coef_abs',
    threshold_number=None
)

# Check how many edges survive per cluster
links.links_dict  # dict {cluster_id: DataFrame with filtered edges}
for cluster, df in links.links_dict.items():
    print(f"Cluster {cluster}: {len(df)} edges after filtering")

# Save and load filtered Links object
#links.to_hdf5(file_path="links.celloracle.links")
#links = co.load_hdf5(file_path="links.celloracle.links")

Cluster ASO_KO: 719593 edges after filtering
Cluster Ectoderm_1: 719593 edges after filtering
Cluster Ectoderm_2: 719593 edges after filtering
Cluster Mesoderm_1: 719593 edges after filtering
Cluster Mesoderm_2: 719593 edges after filtering
Cluster mESCs: 719593 edges after filtering


In [ ]:
# Guardar conteos antes y después
for cluster, df in links.links_dict.items():
    original = links.original_links_count[cluster]  # Asumiendo que existe
    filtrados = len(df)
    eliminados = original - filtrados
    porcentaje = (eliminados / original) * 100
    
    print(f"Cluster {cluster}:")
    print(f"  Original: {original} edges")
    print(f"  Filtrados: {filtrados} edges")
    print(f"  Eliminados: {eliminados} edges ({porcentaje:.1f}%)")
    print()

Basic topology visualization: 

1. undirected degree distribution per cluster

2. centrality scores

In [30]:
# degree distribution plots for the filtered links
links.plot_degree_distributions(
    plot_model=True,
    save=None
)

ASO_KO
Ectoderm_1
Ectoderm_2
Mesoderm_1
Mesoderm_2
mESCs


In [31]:
# Calculate network scores.
links.get_network_score()
links.merged_score.head()

,degree_all,degree_centrality_all,degree_in,degree_centrality_in,degree_out,degree_centrality_out,betweenness_centrality,eigenvector_centrality,cluster
Pou2f2,1,0.001996,0,0.000000,1,0.001996,0.0,4.364361e-14,ASO_KO
Stmn2,1,0.001996,1,0.001996,0,0.000000,0.0,4.356264e-14,ASO_KO
Ebf1,2,0.003992,0,0.000000,2,0.003992,0.0,3.008595e-15,ASO_KO
Hoxa2,1,0.001996,1,0.001996,0,0.000000,0.0,3.365376e-15,ASO_KO
Nhlh1,1,0.001996,0,0.000000,1,0.001996,0.0,7.310698e-01,ASO_KO


Use filtered GRN for simulation:

In [ ]:
oracle.get_cluster_specific_TFdict_from_Links(links_object=links)

oracle.fit_GRN_for_simulation(
    alpha=10,
    use_cluster_specific_TFdict=True  # uses the filtered per-cluster networks
)

## Simulate shift

Also extracts expected expression shift after KO

In [ ]:
# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Rmst": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)


# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
)
axes[0].set_title('RMST1 KO — vector field (grid)')

# Single-cell arrows
oracle.plot_simulation_flow_random_sampling(
    scale=0.4,
    ax=axes[1],
    color_by='leiden',
    n_each_cluster=30
)
axes[1].set_title('RMST1 KO — vector field (single cells)')

plt.tight_layout()
#plt.savefig('../figures/rmst1_ko_vector_field.pdf', dpi=150)
plt.show()


# =============================================================
# SAVE ORACLE OBJECT FOR FURTHER ANALYSIS
# =============================================================

#oracle.to_hdf5("../data/oracle_rmst1_ko.celloracle.hdf5")